# Project PL-Guard: Adversarial Robustness Analysis of Polish NLP Models

#### This project conducts a comprehensive security analysis of BERT-based NLP models when exposed to adversarial text attacks in Polish. Using the PL-Guard dataset, we analyze how malicious modifications (such as typos, leetspeak, and character substitution) affect the semantic representation of text and degrade the performance of safety classifiers.

#### The study moves beyond simple accuracy metrics to explore the geometric distortion of embeddings. We visualize how attacks "shatter" semantic clusters, quantify the "drift" of data points in high-dimensional space, and evaluate whether Adversarial Training can effectively immunize models against these threats

## 1. Data Loading and Basic Preprocessing

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ast
import torch
import plotly.express as px

from transformers import BertTokenizer, BertModel
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE, trustworthiness
from sklearn.metrics import silhouette_score, adjusted_rand_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import plotly.graph_objects as go
from sklearn.ensemble import IsolationForest

import umap


In [ ]:
from datasets import load_dataset

# Load the test dataset
dataset = load_dataset("NASK-PIB/PL-Guard",'test', token=None)

# Load the test_adversarial dataset
dataset_advers = load_dataset("NASK-PIB/PL-Guard", 'test_adversarial', token=None)


##### Short info about what the data set contains, what should be the next steps of preprocessing

In [ ]:
# Convert datasets to pandas DataFrames
df_test = pd.DataFrame(dataset['test'])
df_advers = pd.DataFrame(dataset_advers['test_adversarial'])

# Display basic information about the datasets
print("Test Dataset Info:")
print(df_test.info())
print(df_test.head())
print(df_test.columns)
print("\nAdversarial Test Dataset Info:")
print(df_advers.info())
print(df_advers.head())
print(df_advers.columns)

#### Testing dataset has 2 features with 900 records, they are represented as objects. The dataset with adversarial changes has 5 features with 900 records, the data is also represented as objects

## 2. Text representation Using Embedding
##### For further analysis, the data must be represented numerically. Use simple pretrained embedding model e.g: BERT

##### However, before the text embedding, the texts in the datasets must be cleaned from all unnecessary characters, therefore all special characters and redundant spaces are removed

In [ ]:
def classify_safety(df, column):
    df['category'] = df['category'].astype(str)

    for index, row in df.iterrows():
        if row[column] == 'safe':
            df.at[index, 'category'] = 0
        else:
            cat_number = df.at[index, 'category'].strip('unsafe\nS')
            df.at[index, 'category'] = cat_number
    df['category'] = df['category'].astype(int)
    return df

# Classify safety for both datasets
df_test_numeric = classify_safety(df_test, 'category')
# print(max(df_test_numeric['category'].values))
df_advers_numeric = classify_safety(df_advers, 'category')
# print(max(df_test['category'].values))

# Remove all unnecessary characters and whietespaces from text column
def remove_unnecessary_characters(df, column='text'):
    df[column] = df[column].replace(r'\*\*', '', regex=True)

    df[column] = df[column].replace(r'\"', '', regex=True)

    df[column] = df[column].str.strip()

    df[column] = df[column].replace(r'\s+', ' ', regex=True)
    return df

df_test_cleaned = remove_unnecessary_characters(df_test_numeric)
df_advers_cleaned = remove_unnecessary_characters(df_advers_numeric, column='original_text')
df_advers_cleaned = remove_unnecessary_characters(df_advers_numeric)

# Creating the new columns for mofified texts, firslty modifying the column to a list
df_advers_cleaned['modification_types'] = df_advers_cleaned['modification_types'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

df_advers_cleaned = df_advers_cleaned.explode('modification_types')
count_encoding = pd.crosstab(df_advers_cleaned.index, df_advers_cleaned['modification_types'])
count_encoding.columns = [f'count_{col}' for col in count_encoding.columns]
df_advers_cleaned = df_advers_cleaned.join(count_encoding).fillna(0)

# Drop the column in place
df_advers_cleaned.drop(columns=['modification_types'], inplace=True)

df_advers_cleaned.drop_duplicates(inplace=True)

##### The text is represented numerically using BERT, with Polish model name.

In [ ]:
# Try to find the best available accelerator
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using NVIDIA/ROCm GPU: {torch.cuda.get_device_name(0)}")
else:
    # Check for AMD DirectML (Windows)
    try:
        import torch_directml
        device = torch_directml.device()
        print("Using AMD GPU via DirectML")
    except ImportError:
        device = torch.device("cpu")
        print("Using CPU (No GPU detected. Install 'torch-directml' if on Windows with AMD)")

# Load Model
model_name = "dkleczek/bert-base-polish-uncased-v1"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)
model.to(device)
model.eval()

def get_bert_embeddings(text_list, batch_size=64): 
    all_embeddings = []
    total_samples = len(text_list)
    
    print(f"Starting inference on {device} with Dynamic Padding...")
    
    for i in range(0, total_samples, batch_size):
        if i % (batch_size * 5) == 0:
            print(f"Processing {i}/{total_samples}...")
            
        batch_texts = text_list[i:i+batch_size]

        encoded_batch = tokenizer.batch_encode_plus(
            batch_texts,
            padding=True,      
            truncation=True,
            max_length=512,
            return_tensors='pt'
        )

        input_ids = encoded_batch['input_ids'].to(device)
        attention_mask = encoded_batch['attention_mask'].to(device)

        with torch.no_grad():
            if device.type == 'cuda':
                with torch.amp.autocast('cuda'):
                    outputs = model(input_ids, attention_mask=attention_mask)
            else:
                outputs = model(input_ids, attention_mask=attention_mask)

        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        all_embeddings.append(cls_embeddings.cpu().numpy())

    return np.vstack(all_embeddings)

test_text = df_test_cleaned['text'].tolist()
advers_text = df_advers_cleaned['original_text'].tolist()
advers_modified_text = df_advers_cleaned['text'].tolist()

print("Embedding Test Data...")
df_test_cleaned["embedding"] = list(get_bert_embeddings(test_text, batch_size=32))

print("Embedding Adversarial Original...")
df_advers_cleaned["embedding_original"] = list(get_bert_embeddings(advers_text, batch_size=32))

print("Embedding Adversarial Modified...")
df_advers_cleaned["embedding_modified"] = list(get_bert_embeddings(advers_modified_text, batch_size=32))

## 3. Dimentionality reduction
##### In order to visualize the data, we need to do dimentionality reduction to 2D or 3D. To this end, the **Kmeans, t-SNE and umap** method were used. All visualizations were done using plotly, the "accuracy" of the dimentionality reduction was assessed using trustworthiness method

In [ ]:
X_test = np.vstack(df_test_cleaned["embedding"].values)
y_test = df_test_cleaned["category"].values

X_advers_original = np.vstack(df_advers_cleaned["embedding_original"].values)
X_advers_modified = np.vstack(df_advers_cleaned["embedding_modified"].values)
y_advers = df_advers_cleaned["category"].values


In [ ]:
# --- 1. Prepare and Transform Data ---

# Initialize the Scaler and PCA
scaler = StandardScaler()
pca = PCA(n_components=2)

# A. Process X_test (The Baseline)
# Fit on test data, then transform
X_test_scaled = scaler.fit_transform(X_test)
X_test_pca = pca.fit_transform(X_test_scaled)

# B. Process X_advers (The Attacks)
# Use the SAME scaler and PCA models to transform adversarial data
# (Do NOT fit again, or the axes will mean different things)
X_advers_scaled = scaler.transform(X_advers_modified)
X_advers_pca = pca.transform(X_advers_scaled)

# --- 2. Create Plotting DataFrames ---

# DataFrame for Test Plot
df_plot_test = pd.DataFrame(X_test_pca, columns=['PC1', 'PC2'])
df_plot_test['Category'] = df_test_cleaned['category'].astype(str).values
df_plot_test['Text Snippet'] = df_test_cleaned['text'].astype(str).str.slice(0, 50).values + "..."

# DataFrame for Adversarial Plot
df_plot_advers = pd.DataFrame(X_advers_pca, columns=['PC1', 'PC2'])
df_plot_advers['Category'] = df_advers_cleaned['category'].astype(str).values
df_plot_advers['Text Snippet'] = df_advers_cleaned['text'].astype(str).str.slice(0, 50).values + "..."


# --- 3. Generate the Two Plots ---

# Calculate variance for axis labels
explained_var = pca.explained_variance_ratio_ * 100
labels_map = {
    'PC1': f'PC1 ({explained_var[0]:.2f}% variance)',
    'PC2': f'PC2 ({explained_var[1]:.2f}% variance)'
}

# Plot 1: Clean Test Data
fig1 = px.scatter(
    df_plot_test,
    x='PC1',
    y='PC2',
    color='Category',
    hover_data=['Text Snippet'],
    title='Plot 1: Baseline Clean Prompts (X_test)',
    labels=labels_map,
    opacity=0.7,
    width=800,
    height=600
)

# Plot 2: Adversarial Data
fig2 = px.scatter(
    df_plot_advers,
    x='PC1',
    y='PC2',
    color='Category',
    hover_data=['Text Snippet'],
    title='Plot 2: Adversarial/Modified Prompts (X_advers)',
    labels=labels_map,  # Uses same axis logic as above
    opacity=0.7,
    width=800,
    height=600
)

# Fix axis ranges so they are identical for easier comparison
# (Optional but recommended: forces both plots to have the same zoom level)
all_x = list(df_plot_test['PC1']) + list(df_plot_advers['PC1'])
all_y = list(df_plot_test['PC2']) + list(df_plot_advers['PC2'])
range_x = [min(all_x)*1.1, max(all_x)*1.1]
range_y = [min(all_y)*1.1, max(all_y)*1.1]

fig1.update_xaxes(range=range_x)
fig1.update_yaxes(range=range_y)
fig2.update_xaxes(range=range_x)
fig2.update_yaxes(range=range_y)

# Display
fig1.show()
fig2.show()

In [ ]:
X_combined = np.vstack([X_test_scaled, X_advers_scaled])

# Initialize and Run t-SNE (this might take a moment)
tsne = TSNE(n_components=2, random_state=42, init='pca', learning_rate='auto')
X_combined_tsne = tsne.fit_transform(X_combined)

# Split the results back into Test and Adversarial
len_test = len(X_test_scaled)
X_test_tsne = X_combined_tsne[:len_test]
X_advers_tsne = X_combined_tsne[len_test:]

# --- 2. Create DataFrames ---
df_tsne_test = pd.DataFrame(X_test_tsne, columns=['Dim1', 'Dim2'])
df_tsne_test['Category'] = df_test_cleaned['category'].astype(str).values
df_tsne_test['Text Snippet'] = df_test_cleaned['text'].astype(str).str.slice(0, 50).values + "..."

df_tsne_advers = pd.DataFrame(X_advers_tsne, columns=['Dim1', 'Dim2'])
df_tsne_advers['Category'] = df_advers_cleaned['category'].astype(str).values
df_tsne_advers['Text Snippet'] = df_advers_cleaned['text'].astype(str).str.slice(0, 50).values + "..."

# --- 3. Plotting ---
# Define shared axis range to make comparison valid
all_x = X_combined_tsne[:, 0]
all_y = X_combined_tsne[:, 1]
range_x = [all_x.min()*1.1, all_x.max()*1.1]
range_y = [all_y.min()*1.1, all_y.max()*1.1]

# Plot 1: Clean Test
fig_tsne1 = px.scatter(
    df_tsne_test, x='Dim1', y='Dim2', color='Category',
    hover_data=['Text Snippet'], title='t-SNE: Clean Test Data',
    width=800, height=600, opacity=0.7
)
fig_tsne1.update_xaxes(range=range_x)
fig_tsne1.update_yaxes(range=range_y)

# Plot 2: Adversarial
fig_tsne2 = px.scatter(
    df_tsne_advers, x='Dim1', y='Dim2', color='Category',
    hover_data=['Text Snippet'], title='t-SNE: Adversarial Data',
    width=800, height=600, opacity=0.7
)
fig_tsne2.update_xaxes(range=range_x)
fig_tsne2.update_yaxes(range=range_y)

fig_tsne1.show()
fig_tsne2.show()

In [ ]:
umap_model = umap.UMAP(
    n_components=2,
    n_neighbors=15,   # local vs global balance
    min_dist=0.1,     # cluster tightness
    metric="euclidean",
    random_state=42
)

X_test_umap = umap_model.fit_transform(X_test_scaled)
X_advers_umap = umap_model.transform(X_advers_scaled)

# --- 2. Create DataFrames ---
df_umap_test = pd.DataFrame(X_test_umap, columns=['Dim1', 'Dim2'])
df_umap_test['Category'] = df_test_cleaned['category'].astype(str).values
df_umap_test['Text Snippet'] = df_test_cleaned['text'].astype(str).str.slice(0, 50).values + "..."

df_umap_advers = pd.DataFrame(X_advers_umap, columns=['Dim1', 'Dim2'])
df_umap_advers['Category'] = df_advers_cleaned['category'].astype(str).values
df_umap_advers['Text Snippet'] = df_advers_cleaned['text'].astype(str).str.slice(0, 50).values + "..."

# --- 3. Plotting ---
# Define shared range based on both datasets
all_x = list(X_test_umap[:, 0]) + list(X_advers_umap[:, 0])
all_y = list(X_test_umap[:, 1]) + list(X_advers_umap[:, 1])
range_x = [min(all_x)*1.1, max(all_x)*1.1]
range_y = [min(all_y)*1.1, max(all_y)*1.1]

# Plot 1: Clean Test
fig_umap1 = px.scatter(
    df_umap_test, x='Dim1', y='Dim2', color='Category',
    hover_data=['Text Snippet'], title='UMAP: Clean Test Data',
    width=800, height=600, opacity=0.7
)
fig_umap1.update_xaxes(range=range_x)
fig_umap1.update_yaxes(range=range_y)

# Plot 2: Adversarial
fig_umap2 = px.scatter(
    df_umap_advers, x='Dim1', y='Dim2', color='Category',
    hover_data=['Text Snippet'], title='UMAP: Adversarial Data',
    width=800, height=600, opacity=0.7
)
fig_umap2.update_xaxes(range=range_x)
fig_umap2.update_yaxes(range=range_y)

fig_umap1.show()
fig_umap2.show()

In [ ]:
trust_pca = trustworthiness(X_test_scaled, X_test_pca, n_neighbors=5)
trust_tsne = trustworthiness(X_test_scaled, X_test_tsne, n_neighbors=5)
trust_umap = trustworthiness(X_test_scaled, X_test_umap, n_neighbors=5)

print("Trustworthiness:")
print(f"PCA  : {trust_pca:.4f}")
print(f"t-SNE: {trust_tsne:.4f}")
print(f"UMAP : {trust_umap:.4f}")

##### The **trustworthiness** measures of how many of the k neighbors that are seen as neighbors in the 2D plot, are ACTUALLY neighbors in the original space. The mechanics of PCA simply want to preserve the variance of the data, tehrefore its score may be generally lower. The t-SNE and the UMAP method use more complex appoach, hence their score is better.

## 4. Clustering - unsupervised learning
##### The clustering is performed using: **KMeans, Hierarchical CLustering, DBSCAN and Gaussian Mixture Model**. GMM in contrast to Kmeans, compute the probability of point belonging to one cluster or another, therefore it may prove accurate due to ambiguity of texts

In [ ]:
X_combined_scaled = np.vstack([X_test_scaled, X_advers_scaled])

print("Running t-SNE on combined data (this may take a moment)...")
tsne = TSNE(n_components=2, random_state=42, init='pca', learning_rate='auto')
X_combined_tsne = tsne.fit_transform(X_combined_scaled)

# Split back into Test and Adversarial for plotting
len_test = len(X_test_scaled)
X_test_2d = X_combined_tsne[:len_test]
X_advers_2d = X_combined_tsne[len_test:]

# --- Helper Function for Plotting ---
def plot_clusters(X_2d, labels, df_source, title):
    # Create temp dataframe for plotting
    df_plot = pd.DataFrame(X_2d, columns=['x', 'y'])
    df_plot['Cluster'] = labels.astype(str)
    df_plot['Text'] = df_source['text'].astype(str).str.slice(0, 50).values + "..."
    df_plot['True_Category'] = df_source['category'].astype(str).values

    fig = px.scatter(
        df_plot, x='x', y='y', color='Cluster',
        hover_data=['Text', 'True_Category', 'Cluster'],
        title=title, width=800, height=500, opacity=0.7
    )
    fig.show()

print("t-SNE preparation complete. Ready for clustering cells.")

In [ ]:
# --- K-MEANS CLUSTERING ---

# 1. Fit on Clean Test Data
kmeans = KMeans(n_clusters=15, random_state=42, n_init=10)
labels_test_kmeans = kmeans.fit_predict(X_test_scaled)

# 2. Predict on Adversarial Data (Using the same centers)
labels_advers_kmeans = kmeans.predict(X_advers_scaled)

# 3. Visualize
print(">>> Plotting K-Means Results")
plot_clusters(X_test_2d, labels_test_kmeans, df_test_cleaned, "K-Means: Test Data (15 Clusters)")
plot_clusters(X_advers_2d, labels_advers_kmeans, df_advers_cleaned, "K-Means: Adversarial Data (Predicted)")

In [ ]:
# --- HIERARCHICAL CLUSTERING ---

# 1. Fit on Clean Test Data
agg_clustering = AgglomerativeClustering(n_clusters=15)
labels_test_agg = agg_clustering.fit_predict(X_test_scaled)

# 2. Fit on Adversarial Data (Independent fit)
# Note: Cluster "0" here might not match Cluster "0" above
labels_advers_agg = agg_clustering.fit_predict(X_advers_scaled)

# 3. Visualize
print(">>> Plotting Hierarchical Clustering Results")
plot_clusters(X_test_2d, labels_test_agg, df_test_cleaned, "Hierarchical: Test Data (15 Clusters)")
plot_clusters(X_advers_2d, labels_advers_agg, df_advers_cleaned, "Hierarchical: Adversarial Data")

In [ ]:
# --- DBSCAN CLUSTERING ---

# 1. Fit on Clean Test Data
# eps: max distance between samples to be considered neighbors
# min_samples: min samples in neighborhood to be a core point
dbscan = DBSCAN(eps=22.0, min_samples=3, metric='euclidean') # Adjust eps if you get only -1 (noise)
labels_test_dbscan = dbscan.fit_predict(X_test_scaled)

# 2. Fit on Adversarial Data
# Adversarial data often looks more "noisy" to DBSCAN
labels_advers_dbscan = dbscan.fit_predict(X_advers_scaled)

# Calculate Outlier Stats
outliers_test = list(labels_test_dbscan).count(-1)
outliers_adv = list(labels_advers_dbscan).count(-1)
print(f"Outliers (Noise) in Test: {outliers_test}")
print(f"Outliers (Noise) in Adversarial: {outliers_adv}")

# 3. Visualize
print(">>> Plotting DBSCAN Results (-1 is Noise/Outlier)")
plot_clusters(X_test_2d, labels_test_dbscan, df_test_cleaned, "DBSCAN: Test Data")
plot_clusters(X_advers_2d, labels_advers_dbscan, df_advers_cleaned, "DBSCAN: Adversarial Data")

In [ ]:
from sklearn.mixture import GaussianMixture
# --- GAUSSIAN MIXTURE MODELS (GMM) ---

# 1. Fit on Clean Test Data
# Increase reg_covar from default 1e-6 to 1e-5 or 1e-4
gmm = GaussianMixture(n_components=15, random_state=42, reg_covar=1e-4)
labels_test_gmm = gmm.fit_predict(X_test_scaled)

# 2. Predict on Adversarial Data
labels_advers_gmm = gmm.predict(X_advers_scaled)

# 3. Visualize
print(">>> Plotting GMM Results")
plot_clusters(X_test_2d, labels_test_gmm, df_test_cleaned, "GMM: Test Data (15 Components)")
plot_clusters(X_advers_2d, labels_advers_gmm, df_advers_cleaned, "GMM: Adversarial Data (Predicted)")

In [ ]:
# --- 1. Gather Results ---
# Organize all your labels from previous cells into a dictionary
methods = {
    "K-Means": (labels_test_kmeans, labels_advers_kmeans),
    "Hierarchical": (labels_test_agg, labels_advers_agg),
    "DBSCAN": (labels_test_dbscan, labels_advers_dbscan),
    "GMM": (labels_test_gmm, labels_advers_gmm)
}

# Ground Truth (Actual Categories)
y_test_true = df_test_cleaned['category'].values
y_adv_true = df_advers_cleaned['category'].values

results = []

print("Calculating metrics for all methods (this takes a moment due to Silhouette Score)...")

# --- 2. Calculate Metrics Loop ---
for name, (lbl_test, lbl_adv) in methods.items():
    
    # A. Test Data Metrics
    # Check if valid number of clusters (>1) for Silhouette
    if len(set(lbl_test)) > 1:
        sil_test = silhouette_score(X_test_scaled, lbl_test)
    else:
        sil_test = -1.0 # Invalid
    
    ari_test = adjusted_rand_score(y_test_true, lbl_test)
    
    # B. Adversarial Data Metrics
    if len(set(lbl_adv)) > 1:
        sil_adv = silhouette_score(X_advers_scaled, lbl_adv)
    else:
        sil_adv = -1.0
        
    ari_adv = adjusted_rand_score(y_adv_true, lbl_adv)
    
    # Append
    results.append({
        "Method": name,
        "Test ARI (Accuracy)": ari_test,
        "Test Silhouette (Separation)": sil_test,
        "Adv ARI (Accuracy)": ari_adv,
        "Adv Silhouette (Separation)": sil_adv
    })

# --- 3. Display Table ---
df_clustering_comparison = pd.DataFrame(results).set_index("Method")
df_clustering_comparison = df_clustering_comparison.round(3)

display(df_clustering_comparison)

##### The clustering methods are compared using the **Silhouette score** and the **Adjusted Rand Index (ARI)**. The Silhouette Score evaluates how well data points are clustered by comparing their similarity to points within the same cluster (cohesion) against the nearest neighboring cluster (separation). It ranges from −1 to 1, where higher values indicate better-defined clusters. 
##### The Adjusted Rand Index (ARI) measures the agreement between predicted cluster assignments and true class labels while correcting for chance. It ranges from −1 to 1, where 1 indicates perfect agreement, 0 corresponds to random labeling, and negative values indicate worse-than-random clustering. 
##### In this case, **the scores do not exceed 12%**, which may indicate the poor method selection, however the result may seem sensible while considering text embeddings. The methods perform better on the 'clean' dataset, which is also an expected observation. Across all method, **the Hierarchical Clustering and the GMM appear to perform the best**.

## 5. Outlier detection
##### We can analyse which prompts sit far from the rest, adversarial prompts that move unexpectedly in embedding space and how they affect the representation.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# --- 1. Outlier Detection (Isolation Forest) ---
# We fit on the CLEAN data to learn what "Normal" looks like.
iso_forest = IsolationForest(contamination=0.05, random_state=42)
iso_forest.fit(X_test_scaled)

# Predict (-1 = Outlier, 1 = Normal)
test_outliers = iso_forest.predict(X_test_scaled)
advers_outliers = iso_forest.predict(X_advers_scaled)

# Add to DataFrames for plotting
df_test_cleaned['Outlier_Status'] = pd.Series(test_outliers).map({1: 'Normal', -1: 'Outlier'})
df_advers_cleaned['Outlier_Status'] = pd.Series(advers_outliers).map({1: 'Normal', -1: 'Outlier'})

# --- 2. Calculate Embedding Drift (Movement) ---
# We calculate distance between the Original vs Modified embedding of the SAME prompt
# Ensure we scale the original adversarial embeddings first
X_advers_orig_scaled = scaler.transform(X_advers_original) # Assumes X_advers_original exists
drift_distances = np.linalg.norm(X_advers_scaled - X_advers_orig_scaled, axis=1)

df_advers_cleaned['Drift_Score'] = drift_distances

# --- PLOT 1: Outliers in Clean Test Data ---
# Using X_test_2d (t-SNE coordinates from previous step)
df_plot_test = pd.DataFrame(X_test_2d, columns=['x', 'y'])
df_plot_test['Outlier_Status'] = df_test_cleaned['Outlier_Status'].values
df_plot_test['Text'] = df_test_cleaned['text'].str.slice(0, 50).values + "..."
df_plot_test['Category'] = df_test_cleaned['category'].astype(str).values

fig1 = px.scatter(
    df_plot_test, x='x', y='y',
    color='Outlier_Status',
    color_discrete_map={'Normal': 'lightgrey', 'Outlier': 'red'},
    hover_data=['Text', 'Category'],
    title='1. Outlier Detection: Clean Test Data',
    opacity=0.7, width=900, height=600
)
fig1.show()

# --- PLOT 2: Outliers in Adversarial Data ---
# See if attacks are flagged as outliers
df_plot_adv = pd.DataFrame(X_advers_2d, columns=['x', 'y'])
df_plot_adv['Outlier_Status'] = df_advers_cleaned['Outlier_Status'].values
df_plot_adv['Text'] = df_advers_cleaned['text'].str.slice(0, 50).values + "..."
df_plot_adv['Category'] = df_advers_cleaned['category'].astype(str).values

fig2 = px.scatter(
    df_plot_adv, x='x', y='y',
    color='Outlier_Status',
    color_discrete_map={'Normal': 'lightgrey', 'Outlier': 'red'},
    hover_data=['Text', 'Category'],
    title='2. Outlier Detection: Adversarial Data',
    opacity=0.7, width=900, height=600
)
fig2.show()

# --- PLOT 3: Histogram of Drift (How far did they move?) ---
fig3 = px.histogram(
    df_advers_cleaned,
    x='Drift_Score',
    nbins=50,
    title='3. Distribution of Adversarial Drift (Euclidean Distance)',
    labels={'Drift_Score': 'Distance Moved in Embedding Space'},
    color_discrete_sequence=['purple'],
    width=900, height=500
)
# Add a line for the mean
mean_drift = np.mean(drift_distances)
fig3.add_vline(x=mean_drift, line_dash="dash", line_color="red", annotation_text=f"Mean: {mean_drift:.2f}")
fig3.show()

# --- PLOT 4: Drift Heatmap (Where do the highly distorted prompts land?) ---
# Points are colored by how much they moved. Darker/Yellower = Moved further.
df_plot_adv['Drift_Score'] = drift_distances

fig4 = px.scatter(
    df_plot_adv, x='x', y='y',
    color='Drift_Score',
    color_continuous_scale='Magma', # Dark to Bright
    hover_data=['Text', 'Category', 'Drift_Score'],
    title='4. Adversarial Drift Map (Color = Distance Moved)',
    labels={'Drift_Score': 'Movement Distance'},
    opacity=0.8, width=900, height=600
)
fig4.show()

# --- Analysis Printout ---
print(f"Clean Outliers: {list(test_outliers).count(-1)} / {len(test_outliers)}")
print(f"Adversarial Outliers: {list(advers_outliers).count(-1)} / {len(advers_outliers)}")
print("-" * 30)
print("Top 3 'Most Moved' Prompts:")
top_indices = np.argsort(drift_distances)[-3:][::-1]
for idx in top_indices:
    print(f"\n[Category: {df_advers_cleaned.iloc[idx]['category']}] Drift: {drift_distances[idx]:.4f}")
    print(f"Original: {df_advers_cleaned.iloc[idx]['original_text']}")
    print(f"Modified: {df_advers_cleaned.iloc[idx]['text']}")

##### To analyze the impact of adversarial perturbations in embedding space, we applied an **Isolation Forest** model trained on clean test embeddings to detect anomalous samples. The model identifies points that deviate from the learned distribution of normal data. In parallel, we quantified **embedding drift** by computing the Euclidean distance between the original and adversarial versions of the same prompt in the standardized embedding space.

##### Out of 900 samples, **45 clean prompts** were identified as outliers, compared to **108 adversarial prompts**, indicating that adversarial modifications are substantially more likely to fall outside the normal embedding distribution. Drift analysis further showed that some adversarial prompts moved very far in embedding space (up to ~52 units), despite often preserving partial semantic structure. Visualizations confirmed that highly distorted prompts tend to land in sparse or anomalous regions of the embedding space, highlighting the effectiveness of embedding-based outlier detection for identifying adversarial behavior.


## 6. Impact of Adversarial Changes
##### This section evaluates how adversarial text perturbations affect semantic representations by measuring **embedding drift**, visualizing **movement in latent space**, and assessing **cluster consistency** under attack.


In [ ]:
# --- 1. Calculate Drift (Euclidean Distance) ---
# Ensure we have scaled embeddings for both original and modified
# (Assuming scaler was fit on X_test_cleaned as in previous steps)

# Embeddings of the ADVERSARIAL set (Original Text vs Modified Text)
X_adv_orig_scaled = scaler.transform(np.vstack(df_advers_cleaned["embedding_original"].values))
X_adv_mod_scaled = scaler.transform(np.vstack(df_advers_cleaned["embedding_modified"].values))

# Calculate distance for every prompt
drift_distances = np.linalg.norm(X_adv_mod_scaled - X_adv_orig_scaled, axis=1)
df_advers_cleaned['drift'] = drift_distances

# --- 2. Arrow Plot (PCA) ---
# We project both Original and Modified versions to 2D to draw the lines
pca_2d = PCA(n_components=2)
# Fit on Clean Test (to keep the "map" consistent) and transform adversarial
pca_2d.fit(X_test_scaled)

coords_orig = pca_2d.transform(X_adv_orig_scaled)
coords_mod = pca_2d.transform(X_adv_mod_scaled)

# Create a figure
fig = go.Figure()

# Add lines connecting Original -> Modified
# We only plot a subset (e.g., top 100 with most drift) to avoid clutter
top_drift_indices = np.argsort(drift_distances)[-150:] 

for i in top_drift_indices:
    fig.add_trace(go.Scatter(
        x=[coords_orig[i, 0], coords_mod[i, 0]],
        y=[coords_orig[i, 1], coords_mod[i, 1]],
        mode="lines",
        line=dict(color="red", width=1),
        opacity=0.3,
        showlegend=False
    ))

# Add points for Original (Start)
fig.add_trace(go.Scatter(
    x=coords_orig[top_drift_indices, 0],
    y=coords_orig[top_drift_indices, 1],
    mode="markers",
    marker=dict(color="blue", size=5),
    name="Original Prompt"
))

# Add points for Modified (End)
fig.add_trace(go.Scatter(
    x=coords_mod[top_drift_indices, 0],
    y=coords_mod[top_drift_indices, 1],
    mode="markers",
    marker=dict(color="red", size=5, symbol="x"),
    name="Adversarial Variant"
))

fig.update_layout(
    title="Impact of Attacks: Movement of Top 150 Most Affected Prompts (PCA)",
    xaxis_title="PC1",
    yaxis_title="PC2",
    width=900, height=600
)

fig.show()

In [ ]:
# --- 3. Drift by Modification Type ---

# Identify the 'count_' columns automatically
mod_cols = [col for col in df_advers_cleaned.columns if col.startswith('count_')]

type_stats = []

for col in mod_cols:
    # Filter rows where this specific modification was present
    mask = df_advers_cleaned[col] > 0
    if mask.sum() > 0:
        avg_drift = df_advers_cleaned.loc[mask, 'drift'].mean()
        count = mask.sum()
        # Clean up name (remove 'count_')
        clean_name = col.replace('count_', '')
        type_stats.append({'Modification': clean_name, 'Avg_Drift': avg_drift, 'Count': count})

df_stats = pd.DataFrame(type_stats).sort_values(by='Avg_Drift', ascending=False)

# Visualize Impact
fig_bar = px.bar(
    df_stats,
    x='Modification',
    y='Avg_Drift',
    color='Avg_Drift',
    color_continuous_scale='Reds',
    text_auto='.3f',
    title='Which prompt has changed the most? (Average Embedding Drift by Type)',
    labels={'Avg_Drift': 'Mean Euclidean Distance moved'},
    width=800, height=500
)
fig_bar.show()

print(">>> Summary of Attack Effectiveness")
print(df_stats)

In [ ]:
# --- 4. Cluster Consistency Check ---

# Fit KMeans on Original (Ground Truth)
kmeans = KMeans(n_clusters=15, random_state=42, n_init='auto')
# Fit on the ORIGINALS of the adversarial set
labels_orig = kmeans.fit_predict(X_adv_orig_scaled)

# Predict on the MODIFIED versions
labels_mod = kmeans.predict(X_adv_mod_scaled)

# Check matches
matches = (labels_orig == labels_mod)
success_rate = (~matches).mean() * 100  # % where label CHANGED

print(f"\n>>> Attack Success Rate (Cluster Breaking)")
print(f"Percentage of prompts pushed to a DIFFERENT cluster: {success_rate:.2f}%")

# Show examples where clustering broke
df_advers_cleaned['cluster_orig'] = labels_orig
df_advers_cleaned['cluster_new'] = labels_mod

print("\n--- Examples of Broken Clusters ---")
broken_examples = df_advers_cleaned[df_advers_cleaned['cluster_orig'] != df_advers_cleaned['cluster_new']].head(3)
for idx, row in broken_examples.iterrows():
    print(f"Original Cluster: {row['cluster_orig']} -> New Cluster: {row['cluster_new']}")
    print(f"Text: {row['text']}\n")

##### This experiment evaluates the impact of adversarial text perturbations on semantic embeddings by measuring embedding drift and assessing cluster stability. Embedding drift was computed as the Euclidean distance between standardized embeddings of original and adversarial prompts. To analyze whether this movement translates into semantic inconsistency at a higher level, a K-Means model was trained on the original embeddings and used to evaluate cluster assignment changes after adversarial modification.

##### Across all attack types, adversarial perturbations induced substantial movement in embedding space, with mean drift values consistently around 19–20. Character-level substitution attacks (e.g., substitutions, diacritics, swaps, and keyboard noise) produced the highest average drift, indicating that small but structured character changes are particularly effective at distorting semantic representations. In contrast, deletion-based attacks resulted in the lowest average drift, suggesting a comparatively weaker effect on embedding stability.

##### Cluster consistency analysis revealed that **21.56% of adversarial prompts were reassigned to a different semantic cluster** after modification. This demonstrates that a significant portion of adversarial inputs crossed cluster boundaries, despite retaining largely recognizable meaning to a human reader. Qualitative examples confirm that prompts describing coherent topics such as fraud or election manipulation were pushed into unrelated clusters due to adversarial noise.

##### Overall, these findings show that adversarial text perturbations can induce large semantic shifts in embedding space and meaningfully disrupt clustering structure. This highlights a vulnerability of embedding-based systems, where relatively simple character-level attacks are sufficient to undermine semantic consistency and downstream tasks such as clustering or similarity-based retrieval.

## 7. Classification
##### This experiment evaluates the robustness of simple supervised classifiers for predicting malicious content categories under adversarial text perturbations. We train three standard models—**Logistic Regression**, **Linear SVM**, and **Random Forest**—on clean embeddings and evaluate their performance using accuracy, precision, recall, and confusion-based analysis. The models are then tested on adversarially modified prompts to measure performance degradation and identify vulnerability to attacks. Finally, we investigate whether **adversarial training**, achieved by augmenting the training set with a portion of adversarial samples, can improve robustness against unseen attacks.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# --- 1. DATA SETUP ---

# Clean Data (Source of Truth)
X_clean = X_test
y_clean = df_test_cleaned['category'].values.astype(int)

# Adversarial Data (The Attack)
X_adv = X_advers_modified
y_adv = df_advers_cleaned['category'].values.astype(int)

# Split Clean Data (80% Train, 20% Val)
X_train_clean, X_val_clean, y_train_clean, y_val_clean = train_test_split(
    X_clean, y_clean, test_size=0.2, random_state=42, stratify=y_clean
)

# Split Adversarial Data (50% for Retraining, 50% for Final Testing)
X_adv_train, X_adv_test, y_adv_train, y_adv_test = train_test_split(
    X_adv, y_adv, test_size=0.4, random_state=42, stratify=y_adv
)

# --- 2. DEFINE MODELS ---
models = {
    "Logistic Regression": LogisticRegression(max_iter=3000, random_state=42),
    "SVM (Linear)": SVC(kernel='linear', random_state=42), # Linear works well for high-dim text
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

# --- 3. EXPERIMENT A: TRAIN ON CLEAN ONLY ---
results_clean_training = []

print(">>> RUNNING EXPERIMENT A: Training on Clean Data...")

for name, model in models.items():
    # Train
    model.fit(X_train_clean, y_train_clean)
    
    # Predict on Clean Validation
    y_pred_val = model.predict(X_val_clean)
    
    # Predict on Adversarial Test (The Attack)
    y_pred_adv = model.predict(X_adv_test)
    
    # Calculate Metrics
    results_clean_training.append({
        "Model": name,
        # Clean Metrics
        "Clean Acc": accuracy_score(y_val_clean, y_pred_val),
        "Clean Precision": precision_score(y_val_clean, y_pred_val, average='weighted', zero_division=0),
        "Clean Recall": recall_score(y_val_clean, y_pred_val, average='weighted', zero_division=0),
        # Adversarial Metrics
        "Adv Acc": accuracy_score(y_adv_test, y_pred_adv),
        "Adv Precision": precision_score(y_adv_test, y_pred_adv, average='weighted', zero_division=0),
        "Adv Recall": recall_score(y_adv_test, y_pred_adv, average='weighted', zero_division=0),
        # The Drop
        "Acc Drop (%)": (accuracy_score(y_val_clean, y_pred_val) - accuracy_score(y_adv_test, y_pred_adv)) * 100
    })

# --- 4. EXPERIMENT B: ADVERSARIAL TRAINING (MIXED) ---
results_mixed_training = []

# Create Mixed Dataset (Clean Train + 50% Adversarial)
X_train_mixed = np.vstack([X_train_clean, X_adv_train])
y_train_mixed = np.concatenate([y_train_clean, y_adv_train])

print("\n>>> RUNNING EXPERIMENT B: Retraining on Mixed Data (Adversarial Training)...")

for name, model in models.items():
    # Retrain on Mixed Data
    model.fit(X_train_mixed, y_train_mixed)
    
    # Predict on the UNSEEN Adversarial Test set
    y_pred_robust = model.predict(X_adv_test)
    
    # Compare with previous (vulnerable) performance
    # Find previous score from first experiment
    prev_acc = next(item["Adv Acc"] for item in results_clean_training if item["Model"] == name)
    
    new_acc = accuracy_score(y_adv_test, y_pred_robust)
    
    results_mixed_training.append({
        "Model": name,
        "Original Adv Acc": prev_acc,
        "Robust Adv Acc": new_acc,
        "Improvement (%)": (new_acc - prev_acc) * 100,
        "Robust Precision": precision_score(y_adv_test, y_pred_robust, average='weighted', zero_division=0),
        "Robust Recall": recall_score(y_adv_test, y_pred_robust, average='weighted', zero_division=0)
    })

# --- 5. GENERATE TABLES ---
df_table_1 = pd.DataFrame(results_clean_training).round(3)
df_table_2 = pd.DataFrame(results_mixed_training).round(3)

print("\n" + "="*80)
print("TABLE 1: VULNERABILITY ANALYSIS (Trained on Clean Data Only)")
print("Comparing performance on Clean vs. Adversarial Test Sets")
print("="*80)
display(df_table_1)

print("\n" + "="*80)
print("TABLE 2: DEFENSE ANALYSIS (Adversarial Training)")
print("Did adding 50% of attacks to training improve detection on unseen attacks?")
print("="*80)
display(df_table_2)

# --- 6. VISUALIZATION OF IMPROVEMENT ---
fig = px.bar(
    df_table_2,
    x="Model",
    y=["Original Adv Acc", "Robust Adv Acc"],
    barmode='group',
    title="Effect of Adversarial Training: Accuracy Before vs After",
    labels={"value": "Accuracy on Adversarial Prompts", "variable": "Training Strategy"},
    text_auto='.2%'
)
fig.show()

# --- 7. CONFUSION MATRIX FOR BEST ROBUST MODEL ---
# Pick best model from Table 2
best_model_name = df_table_2.sort_values("Robust Adv Acc", ascending=False).iloc[0]["Model"]
print(f"\n>>> Best Robust Model: {best_model_name}")

##### Baseline Performance and Vulnerability to Adversarial Attacks

When trained exclusively on clean data, all models achieve moderate classification performance on the clean validation set. Logistic Regression and Linear SVM perform comparably, while Random Forest shows lower overall accuracy, reflecting the challenges of tree-based methods in high-dimensional embedding spaces.

Evaluation on adversarial prompts reveals varying levels of robustness. Logistic Regression experiences a noticeable accuracy drop of **7.22%**, indicating sensitivity to adversarial perturbations. Random Forest shows a smaller degradation (**1.94%**), but its absolute performance remains the weakest. In contrast, the Linear SVM exhibits **no measurable accuracy drop**, suggesting that its linear decision boundaries are more stable under the applied character-level attacks.

##### Adversarial Training as a Defense

To improve robustness, models were retrained using a mixed dataset consisting of clean data augmented with 50% of the adversarial samples. This adversarial training strategy significantly improves performance for linear models. Logistic Regression shows the largest gain, with adversarial accuracy increasing from **0.661 to 0.764** (+10.28%), while Linear SVM improves from **0.728 to 0.769** (+4.17%). These results demonstrate that exposure to adversarial patterns during training enables the models to learn more robust decision boundaries.

In contrast, Random Forest performance slightly degrades after adversarial training, suggesting that it struggles to generalize from noisy adversarial examples and may overfit to perturbation artifacts rather than semantic structure.

Overall, the results indicate that adversarial text perturbations can significantly degrade classification performance, particularly for models trained only on clean data. However, **adversarial training proves effective for linear classifiers**, substantially improving robustness on unseen attacks. The Linear SVM emerges as the most stable model across both clean and adversarial settings, while Logistic Regression benefits the most from adversarial augmentation. These findings highlight the importance of incorporating adversarial examples during training when deploying embedding-based classifiers in adversarial or noisy environments.